In [ ]:
!pip install catboost -q

In [ ]:
import os
import sys
import joblib
import logging
import numpy as np
import pandas as pd
import rasterio

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)

logging.info("Logging is working ✅")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/PhytoLB"

GBIF_PATH = f"{BASE_DIR}/dwca-tree_lebanon-v1.0/occurrence.txt"
CLIMATE_DIR = f"{BASE_DIR}/wc2.1_10m_bio"
ELEVATION_PATH = f"{BASE_DIR}/elevation/elevation.tif"

SAVE_DIR = f"{BASE_DIR}/model_catboost_v1"
os.makedirs(SAVE_DIR, exist_ok=True)

logging.info(f"Saving outputs to: {SAVE_DIR}")

In [ ]:
def load_gbif_txt(path):
    df = pd.read_csv(path, sep="\t", low_memory=False)

    if "species" not in df.columns:
        df["species"] = df["scientificName"]

    df = df[["species", "decimalLatitude", "decimalLongitude"]]
    df = df.dropna()

    df.columns = ["species", "lat", "lon"]

    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna()

    df = df[
        (df["lat"] >= 33.0) & (df["lat"] <= 34.8) &
        (df["lon"] >= 35.0) & (df["lon"] <= 36.8)
    ]

    df = df.drop_duplicates()

    logging.info(f"GBIF loaded: {len(df)} records")
    logging.info(f"Unique species: {df['species'].nunique()}")

    return df

In [ ]:
class EnvironmentalExtractor:
    def __init__(self, climate_folder, elevation_path):
        self.climate_rasters = []

        for i in range(1, 20):
            path = os.path.join(climate_folder, f"wc2.1_10m_bio_{i}.tif")

            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing climate file: {path}")

            self.climate_rasters.append(rasterio.open(path))

        if not os.path.exists(elevation_path):
            raise FileNotFoundError(f"Missing elevation file: {elevation_path}")

        self.elevation_raster = rasterio.open(elevation_path)

        logging.info("Climate + elevation rasters loaded ✅")

    def extract_point(self, lat, lon):
        values = []

        for raster in self.climate_rasters:
            try:
                val = next(raster.sample([(lon, lat)]))[0]

                if raster.nodata is not None and val == raster.nodata:
                    val = np.nan

            except Exception:
                val = np.nan

            values.append(val)

        try:
            elev = next(self.elevation_raster.sample([(lon, lat)]))[0]

            if self.elevation_raster.nodata is not None and elev == self.elevation_raster.nodata:
                elev = np.nan

        except Exception:
            elev = np.nan

        values.append(elev)

        return values

    def extract_dataframe(self, df):
        features = []
        total = len(df)

        for i, row in enumerate(df.itertuples(index=False)):
            values = self.extract_point(row.lat, row.lon)
            features.append(values)

            if i % 1000 == 0:
                logging.info(f"Processed {i}/{total} points")

        columns = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

        return pd.DataFrame(features, columns=columns)

In [ ]:
def add_engineered_features(df):
    df = df.copy()

    df["temp_annual_range"] = df["BIO5"] - df["BIO6"]
    df["warm_cold_quarter_diff"] = df["BIO10"] - df["BIO11"]

    df["precip_wet_dry_month_diff"] = df["BIO13"] - df["BIO14"]
    df["precip_wet_dry_quarter_diff"] = df["BIO16"] - df["BIO17"]

    df["dryness_ratio"] = df["BIO17"] / (df["BIO16"] + 1e-6)
    df["summer_winter_precip_ratio"] = df["BIO18"] / (df["BIO19"] + 1e-6)

    df["elevation_temp_interaction"] = df["elevation"] * df["BIO1"]
    df["elevation_precip_interaction"] = df["elevation"] * df["BIO12"]

    return df


In [ ]:
def generate_pseudo_absence(df, n_samples, extractor, random_state=42):
    np.random.seed(random_state)

    lat_min, lat_max = df["lat"].min(), df["lat"].max()
    lon_min, lon_max = df["lon"].min(), df["lon"].max()

    pseudo_rows = []
    attempts = 0
    max_attempts = n_samples * 20

    logging.info("Generating filtered pseudo-absence points...")

    while len(pseudo_rows) < n_samples and attempts < max_attempts:
        attempts += 1

        lat = np.random.uniform(lat_min, lat_max)
        lon = np.random.uniform(lon_min, lon_max)

        values = extractor.extract_point(lat, lon)

        if np.isnan(values).sum() > 0:
            continue

        elevation = values[-1]

        if elevation < 0 or elevation > 3200:
            continue

        species = df["species"].sample(
            1,
            random_state=np.random.randint(0, 999999)
        ).iloc[0]

        pseudo_rows.append({
            "species": species,
            "lat": lat,
            "lon": lon,
            "label": 0
        })

        if len(pseudo_rows) % 1000 == 0:
            logging.info(f"Generated {len(pseudo_rows)}/{n_samples} valid pseudo-absences")

    pseudo_df = pd.DataFrame(pseudo_rows)

    if len(pseudo_df) < n_samples:
        logging.warning(f"Only generated {len(pseudo_df)} pseudo-absences out of {n_samples}")

    return pseudo_df

In [ ]:
def build_dataset(df, extractor):
    logging.info("Extracting presence features...")

    presence_features = extractor.extract_dataframe(df)
    presence_features["species"] = df["species"].values
    presence_features["lat"] = df["lat"].values
    presence_features["lon"] = df["lon"].values
    presence_features["label"] = 1

    pseudo_df = generate_pseudo_absence(
        df=df,
        n_samples=len(df),
        extractor=extractor,
        random_state=42
    )

    logging.info("Extracting pseudo-absence features...")

    absence_features = extractor.extract_dataframe(pseudo_df)
    absence_features["species"] = pseudo_df["species"].values
    absence_features["lat"] = pseudo_df["lat"].values
    absence_features["lon"] = pseudo_df["lon"].values
    absence_features["label"] = 0

    full_df = pd.concat([presence_features, absence_features], ignore_index=True)

    logging.info(f"Dataset before filtering: {full_df.shape}")

    env_cols = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]
    full_df = full_df.dropna(subset=env_cols)

    full_df = add_engineered_features(full_df)

    logging.info(f"Dataset after feature engineering: {full_df.shape}")

    return full_df

In [ ]:
def suitability_class(prob):
    if prob >= 0.75:
        return "Highly suitable"
    elif prob >= 0.50:
        return "Suitable"
    elif prob >= 0.30:
        return "Moderately suitable"
    else:
        return "Low / not suitable"


def find_species(name_part):
    matches = [sp for sp in le.classes_ if name_part.lower() in sp.lower()]

    if not matches:
        print("No species found.")
    else:
        for sp in matches:
            print(sp)

In [ ]:
gbif = load_gbif_txt(GBIF_PATH)
gbif.to_csv(f"{SAVE_DIR}/clean_gbif.csv", index=False)

species_counts = gbif["species"].value_counts().reset_index()
species_counts.columns = ["species", "records"]
species_counts.to_csv(f"{SAVE_DIR}/species_counts.csv", index=False)


extractor = EnvironmentalExtractor(
    climate_folder=CLIMATE_DIR,
    elevation_path=ELEVATION_PATH
)


dataset = build_dataset(gbif, extractor)

logging.info(f"Final dataset shape: {dataset.shape}")
logging.info(f"Class distribution:\n{dataset['label'].value_counts()}")
logging.info(f"Missing values total: {dataset.isna().sum().sum()}")

dataset.to_csv(f"{SAVE_DIR}/catboost_phyto_dataset.csv", index=False)
dataset.to_parquet(f"{SAVE_DIR}/catboost_phyto_dataset.parquet", index=False)


X = dataset.drop(columns=["label"])
y = dataset["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logging.info(f"Train size: {X_train.shape}")
logging.info(f"Test size: {X_test.shape}")


X_train = X_train.copy()
X_test = X_test.copy()

X_train[["species", "lat", "lon"]].to_csv(f"{SAVE_DIR}/train_diagnostics.csv", index=False)
X_test[["species", "lat", "lon"]].to_csv(f"{SAVE_DIR}/test_diagnostics.csv", index=False)

X_train = X_train.drop(columns=["lat", "lon"])
X_test = X_test.drop(columns=["lat", "lon"])

cat_features = ["species"]

feature_names = X_train.columns.tolist()

logging.info(f"Feature count: {len(feature_names)}")
logging.info(f"Categorical features: {cat_features}")


numeric_features = [col for col in feature_names if col not in cat_features]

imputer = SimpleImputer(strategy="median")

X_train[numeric_features] = imputer.fit_transform(X_train[numeric_features])
X_test[numeric_features] = imputer.transform(X_test[numeric_features])

logging.info(f"NaNs train: {X_train.isna().sum().sum()}")
logging.info(f"NaNs test: {X_test.isna().sum().sum()}")

X_train.to_csv(f"{SAVE_DIR}/X_train_catboost_processed.csv", index=False)
X_test.to_csv(f"{SAVE_DIR}/X_test_catboost_processed.csv", index=False)
y_train.to_csv(f"{SAVE_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{SAVE_DIR}/y_test.csv", index=False)


cat_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.025,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    l2_leaf_reg=3,
    subsample=0.9,
    bootstrap_type="Bernoulli",
    verbose=100
)

logging.info("Training CatBoost model...")

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

logging.info("CatBoost training completed ✅")


cat_model.save_model(f"{SAVE_DIR}/catboost_phyto_model.cbm")
joblib.dump(cat_model, f"{SAVE_DIR}/catboost_phyto_model.pkl")
joblib.dump(imputer, f"{SAVE_DIR}/numeric_imputer.pkl")
joblib.dump(feature_names, f"{SAVE_DIR}/feature_names.pkl")
joblib.dump(cat_features, f"{SAVE_DIR}/cat_features.pkl")
joblib.dump(numeric_features, f"{SAVE_DIR}/numeric_features.pkl")

logging.info("CatBoost model + artifacts saved ✅")

y_pred = cat_model.predict(X_test)
y_prob = cat_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

logging.info(f"CatBoost Accuracy: {acc:.4f}")
logging.info(f"CatBoost AUC: {auc:.4f}")

print("\nClassification Report:\n")
print(report)

print("\nConfusion Matrix:")
print(cm)

with open(f"{SAVE_DIR}/catboost_evaluation_report.txt", "w") as f:
    f.write(f"CatBoost Accuracy: {acc:.4f}\n")
    f.write(f"CatBoost AUC: {auc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(report)
    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))


importance_values = cat_model.get_feature_importance()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance_values
}).sort_values("importance", ascending=False)

importance_df.to_csv(f"{SAVE_DIR}/catboost_feature_importance.csv", index=False)

print("\nTop 15 CatBoost Features:")
print(importance_df.head(15))


def predict_species_suitability_catboost(city_name, lat, lon, species_name):
    if species_name not in gbif["species"].unique():
        print(f"'{species_name}' not found.")
        print("Closest matches:")
        for sp in sorted(gbif["species"].unique()):
            if species_name.split()[0].lower() in sp.lower():
                print("-", sp)
        return None

    env_values = extractor.extract_point(lat, lon)

    base_cols = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

    sample_df = pd.DataFrame([env_values], columns=base_cols)
    sample_df = add_engineered_features(sample_df)

    sample_df["species"] = species_name

    # Match training columns
    sample_df = sample_df[feature_names]

    sample_df[numeric_features] = imputer.transform(sample_df[numeric_features])

    prob = cat_model.predict_proba(sample_df)[0][1]
    label = int(prob >= 0.5)

    print("====================================")
    print(f"City: {city_name}")
    print(f"Species: {species_name}")
    print(f"Suitability Probability: {prob:.4f}")
    print(f"Suitability Class: {suitability_class(prob)}")
    print(f"Binary Prediction: {'Suitable' if label == 1 else 'Not suitable'}")
    print("====================================")

    return float(prob), label


def batch_predict_catboost(cities, species_list):
    results = []

    for city_name, lat, lon in cities:
        for species_name in species_list:
            output = predict_species_suitability_catboost(
                city_name,
                lat,
                lon,
                species_name
            )

            if output is None:
                continue

            prob, label = output

            results.append({
                "city": city_name,
                "lat": lat,
                "lon": lon,
                "species": species_name,
                "probability": prob,
                "class": suitability_class(prob),
                "binary_label": label
            })

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{SAVE_DIR}/catboost_city_species_predictions.csv", index=False)

    return results_df


test_cities = [
    ("Beirut", 33.8938, 35.5018),
    ("Tripoli", 34.4367, 35.8497),
    ("Byblos", 34.1230, 35.6519),
    ("Bcharre", 34.2508, 36.0106),
    ("Cedars of God", 34.2436, 36.0486),
    ("Faraya", 33.9944, 35.8172),
    ("Zahle", 33.8463, 35.9020),
    ("Baalbek", 34.0060, 36.2181),
    ("Sidon", 33.5630, 35.3688),
    ("Tyre", 33.2700, 35.2038)
]

test_species = [
    "Cedrus libani",
    "Ceratonia siliqua",
    "Abies cilicica",
    "Quercus coccifera calliprinos",
    "Quercus infectoria boissieri",
    "Arbutus andrachne",
    "Juniperus excelsa",
    "Juniperus drupacea"
]

predictions_df = batch_predict_catboost(test_cities, test_species)

print("\nPrediction summary:")
print(predictions_df.sort_values("probability", ascending=False).head(20))

logging.info("CatBoost pipeline completed successfully 🚀")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00
2026-05-10 06:40:48,214 - INFO - Logging is working ✅
2026-05-10 06:40:48,222 - INFO - Saving outputs to: /content/drive/MyDrive/PhytoLB/model_catboost_v1
2026-05-10 06:40:48,292 - INFO - GBIF loaded: 13748 records
2026-05-10 06:40:48,294 - INFO - Unique species: 27
2026-05-10 06:40:48,438 - INFO - Climate + elevation rasters loaded ✅
2026-05-10 06:40:48,443 - INFO - Extracting presence features...
2026-05-10 06:40:48,459 - INFO - Processed 0/13748 points
2026-05-10 06:40:51,245 - INFO - Processed 1000/13748 points
2026-05-10 06:40:55,226 - INFO - Processed 2000/13748 points
2026-05-10 06:40:58,130 - INFO - Processed 3000/13748 points
2026-05-10 06:41:00,975 - INFO - Processed 4000/13748 points
2026-05-10 06:41:03,756 - INFO - Processed 5000/13748 points
2026-05-10 06:41:06,920 - INFO - Processed 6000/13748 points
2026-05-10 06:41:10,823 - INFO - Processed 7000/13748 points
2026-05-10 06:41:13,483 - INFO - Pr